# Análise Bienal — FAMAR & FUMES
### Acidentes por Trimestre · Teste Exato de Fisher (Gênero)

In [1]:
# ─── Instalação de dependências ───────────────────────────────────────────────
# scipy já vem no Colab; instalação explícita caso rode em outro ambiente
# !pip install scipy pandas --quiet

In [4]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from google.colab import drive

In [7]:
# ─── 3. LEITURA E IDENTIFICAÇÃO DAS PLANILHAS ─────────────────────────────────
# ⚙️  Ajuste PASTA_DRIVE para o caminho da pasta no seu Drive onde estão os CSVs.
# Exemplo: 'Meu Drive/dados_acidentes'  ou  'Meu Drive'  se estiverem na raiz.
PASTA_DRIVE = 'Meu Drive'   # ← altere aqui se necessário

ARQUIVOS_ESPERADOS = [
    '2023_famar.csv',
    '2024_famar.csv',
    '2023_fumes.csv',
    '2024_fumes.csv',
]

def identificar_instituicao(nome_arquivo: str) -> str:
    nome = nome_arquivo.lower()
    if 'fumes' in nome:
        return 'FUMES'
    elif 'famar' in nome:
        return 'FAMAR'
    raise ValueError(f'Não foi possível identificar a instituição em: {nome_arquivo}')

planilhas = {}   # {'nome_arquivo': DataFrame}

for nome in ARQUIVOS_ESPERADOS:
    caminho = f'{nome}'
    try:
        df = pd.read_csv(caminho)
    except FileNotFoundError:
        print(f'[AVISO] Arquivo não encontrado: {caminho}')
        continue

    df['instituicao'] = identificar_instituicao(nome)
    df['arquivo_origem'] = nome

    # Converter coluna de data (formato yyyy-dd-MM hh:mm:ss)
    # Tenta converter no formato com hora
    # Tenta converter no formato com hora
    datas_convertidas = pd.to_datetime(
        df['data_do_acidente'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    # Onde falhou, tenta no formato sem hora
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d',
            errors='coerce'
        )
    )

    # Imprime datas inválidas
    mask_invalidas = (
        datas_convertidas.isna()
        & df['data_do_acidente'].notna()
    )

    if mask_invalidas.any():
        print('\nDatas com erro de conversão:')

        for idx, valor in df.loc[
            mask_invalidas,
            'data_do_acidente'
        ].items():

            print(
                f'  Linha {idx + 2}: '
                f'"{valor}" '
                f'(esperado: %Y-%m-%d %H:%M:%S '
                f'ou %Y-%m-%d)'
            )

    # Salva no dataframe
    df['data_do_acidente'] = datas_convertidas

    planilhas[nome] = df
    print(f'[OK] {nome} — {len(df)} registros | instituição: {df["instituicao"].iloc[0]}')

print(f'\nTotal de arquivos carregados: {len(planilhas)}')

[OK] 2023_famar.csv — 93 registros | instituição: FAMAR
[OK] 2024_famar.csv — 83 registros | instituição: FAMAR
[OK] 2023_fumes.csv — 9 registros | instituição: FUMES
[OK] 2024_fumes.csv — 12 registros | instituição: FUMES

Total de arquivos carregados: 4


In [8]:
# ─── 4. AGREGAÇÃO DOS BIÊNIOS ─────────────────────────────────────────────────
# Biênio FAMAR (2023 + 2024)
df_famar = pd.concat(
    [planilhas[f] for f in ['2023_famar.csv', '2024_famar.csv'] if f in planilhas],
    ignore_index=True
)

# Biênio FUMES (2023 + 2024)
df_fumes = pd.concat(
    [planilhas[f] for f in ['2023_fumes.csv', '2024_fumes.csv'] if f in planilhas],
    ignore_index=True
)

print(f'Biênio FAMAR — total de registros: {len(df_famar)}')
print(f'Biênio FUMES — total de registros: {len(df_fumes)}')

Biênio FAMAR — total de registros: 176
Biênio FUMES — total de registros: 21


In [12]:
import pandas as pd

FORMATO_DATA = '%Y-%m-%d %H:%M:%S'

print('=' * 60)
print('  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA')
print('=' * 60)

total_geral = 0

for nome, df in planilhas.items():
    n = len(df)
    total_geral += n

    inst = (
        df['instituicao'].iloc[0]
        if not df.empty and 'instituicao' in df.columns
        else 'N/D'
    )

    datas_nulas = df['data_do_acidente'].isna().sum()
    datas_invalidas = []

    for idx, val in df['data_do_acidente'].items():

        # Ignora nulos
        if pd.isna(val):
            continue

        linha_planilha = idx + 2  # +1 header +1 índice zero-based
        valor_original = str(val)

        # try:
        #     pd.to_datetime(
        #         valor_original,
        #         format=FORMATO_DATA,
        #         errors='raise'
        #     )

        # except Exception:

        #     motivo = (
        #         'Formato inválido. '
        #         'Esperado: yyyy-mm-dd HH:MM:SS '
        #         f'(ex.: 2023-05-31 00:00:00)'
        #     )

        #     datas_invalidas.append({
        #         'linha': linha_planilha,
        #         'valor': valor_original,
        #         'motivo': motivo
        #     })

    print(f'\nArquivo : {nome}')
    print(f'  Instituição       : {inst}')
    print(f'  Registros totais  : {n}')
    print(f'  Datas nulas       : {datas_nulas}')
    print(f'  Datas inválidas   : {len(datas_invalidas)}')

    if datas_invalidas:
        print('\n  Datas fora do formato:')

        print(
            f'  {"Linha":>6}  '
            f'{"Valor encontrado":<30}  '
            f'Motivo'
        )

        print(
            f'  {"-" * 6}  '
            f'{"-" * 30}  '
            f'{"-" * 60}'
        )

        for erro in datas_invalidas:
            print(
                f'  {erro["linha"]:>6}  '
                f'{erro["valor"]:<30}  '
                f'{erro["motivo"]}'
            )

    else:
        print('  Datas fora de formato : nenhuma')

print(f'\n{" TOTAL GERAL ":=^60}')
print(
    f'  {total_geral} registros importados '
    f'em {len(planilhas)} arquivos'
)

  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA

Arquivo : 2023_famar.csv
  Instituição       : FAMAR
  Registros totais  : 93
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_famar.csv
  Instituição       : FAMAR
  Registros totais  : 83
  Datas nulas       : 1
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2023_fumes.csv
  Instituição       : FUMES
  Registros totais  : 9
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_fumes.csv
  Instituição       : FUMES
  Registros totais  : 12
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

======================= TOTAL GERAL ========================
  197 registros importados em 4 arquivos


In [15]:
# ─── 5. FUNÇÃO: TRIMESTRE A PARTIR DA DATA ────────────────────────────────────
def adicionar_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['trimestre'] = (
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        .dt.quarter
        .apply(lambda x: f'T{x}' if pd.notna(x) else None)
    )

    return df


df_famar = adicionar_trimestre(df_famar)
df_fumes = adicionar_trimestre(df_fumes)

print(
    'Trimestres identificados — FAMAR:',
    sorted(df_famar['trimestre'].dropna().unique())
)

print(
    'Trimestres identificados — FUMES:',
    sorted(df_fumes['trimestre'].dropna().unique())
)

Trimestres identificados — FAMAR: ['T1.0', 'T2.0', 'T3.0', 'T4.0']
Trimestres identificados — FUMES: ['T1', 'T2', 'T3', 'T4']


In [16]:
# ─── 7. RELATÓRIO TRIMESTRAL DOS BIÊNIOS ──────────────────────────────────────
def relatorio_trimestral(df: pd.DataFrame, nome_inst: str):
    total = len(df)
    print(f'\n{" " + nome_inst + " — Biênio por Trimestre ":=^60}')
    print(f'  Total do biênio: {total} registros\n')

    por_trim = (
        df.groupby('trimestre', dropna=False)
          .size()
          .reset_index(name='n')
          .sort_values('trimestre')
    )

    print(f'  {"Trimestre":<15} {"N (abs)":>10} {"% do biênio":>14}')
    print(f'  {"-"*15} {"-"*10} {"-"*14}')

    for _, row in por_trim.iterrows():
        trim = str(row['trimestre']) if pd.notna(row['trimestre']) else 'Data inválida'
        n    = int(row['n'])
        pct  = (n / total * 100) if total > 0 else 0
        print(f'  {trim:<15} {n:>10} {pct:>13.1f}%')

    print(f'  {"TOTAL":<15} {total:>10} {100.0:>13.1f}%')

relatorio_trimestral(df_famar, 'FAMAR')
relatorio_trimestral(df_fumes, 'FUMES')


=============== FAMAR — Biênio por Trimestre ===============
  Total do biênio: 176 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1.0                    49          27.8%
  T2.0                    35          19.9%
  T3.0                    52          29.5%
  T4.0                    39          22.2%
  Data inválida            1           0.6%
  TOTAL                  176         100.0%

=============== FUMES — Biênio por Trimestre ===============
  Total do biênio: 21 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1                       5          23.8%
  T2                       6          28.6%
  T3                       7          33.3%
  T4                       3          14.3%
  TOTAL                   21         100.0%


In [17]:
# ─── 8. TESTE EXATO DE FISHER — GÊNERO ───────────────────────────────────────
# Normalização dos valores de gênero
def normalizar_genero(series: pd.Series) -> pd.Series:
    return series.str.strip().str.lower().map(
        lambda x: 'masculino' if str(x) == 'masculino'
        else ('feminino' if str(x) == 'feminino' else np.nan)
    )

def contar_genero(df: pd.DataFrame) -> tuple[int, int]:
    gen = normalizar_genero(df['genero'])
    masc = (gen == 'masculino').sum()
    fem  = (gen == 'feminino').sum()
    return int(masc), int(fem)

masc_famar, fem_famar = contar_genero(df_famar)
masc_fumes, fem_fumes = contar_genero(df_fumes)

# Tabela de contingência 2x2
# Linhas: instituição (FAMAR / FUMES)
# Colunas: gênero (masculino / feminino)
tabela_contingencia = np.array([
    [masc_famar, fem_famar],
    [masc_fumes, fem_fumes]
])

odds_ratio, p_valor = fisher_exact(tabela_contingencia, alternative='two-sided')

print('=' * 60)
print('  TESTE EXATO DE FISHER — GÊNERO (Masculino vs Feminino)')
print('=' * 60)

print('\n  Tabela de Contingência 2×2:')
print(f'  {"":20} {"Masculino":>12} {"Feminino":>12} {"Total":>10}')
print(f'  {"-"*56}')

total_famar_gen = masc_famar + fem_famar
total_fumes_gen = masc_fumes + fem_fumes
total_masc = masc_famar + masc_fumes
total_fem  = fem_famar  + fem_fumes
total_geral_gen = total_famar_gen + total_fumes_gen

def fmt(n, total):
    pct = (n / total * 100) if total > 0 else 0
    return f'{n} ({pct:.1f}%)'

print(f'  {"FAMAR":<20} {fmt(masc_famar, total_famar_gen):>22} {fmt(fem_famar, total_famar_gen):>22} {total_famar_gen:>10}')
print(f'  {"FUMES":<20} {fmt(masc_fumes, total_fumes_gen):>22} {fmt(fem_fumes, total_fumes_gen):>22} {total_fumes_gen:>10}')
print(f'  {"-"*56}')
print(f'  {"TOTAL":<20} {total_masc:>22} {total_fem:>22} {total_geral_gen:>10}')

print('\n  Números absolutos:')
print(f'    FAMAR — Masculino: {masc_famar:>6} | Feminino: {fem_famar:>6}')
print(f'    FUMES — Masculino: {masc_fumes:>6} | Feminino: {fem_fumes:>6}')

print('\n  Resultados do Teste Exato de Fisher (two-sided):')
print(f'    Odds Ratio : {odds_ratio:.4f}')
print(f'    p-valor    : {p_valor:.6f}')

ALPHA = 0.05
if p_valor < ALPHA:
    print(f'\n  Conclusão: Diferença estatisticamente SIGNIFICATIVA')
    print(f'  (p = {p_valor:.6f} < α = {ALPHA})')
else:
    print(f'\n  Conclusão: Diferença NÃO significativa estatisticamente')
    print(f'  (p = {p_valor:.6f} ≥ α = {ALPHA})')

  TESTE EXATO DE FISHER — GÊNERO (Masculino vs Feminino)

  Tabela de Contingência 2×2:
                          Masculino     Feminino      Total
  --------------------------------------------------------
  FAMAR                            35 (19.9%)            141 (80.1%)        176
  FUMES                              1 (4.8%)             20 (95.2%)         21
  --------------------------------------------------------
  TOTAL                                    36                    161        197

  Números absolutos:
    FAMAR — Masculino:     35 | Feminino:    141
    FUMES — Masculino:      1 | Feminino:     20

  Resultados do Teste Exato de Fisher (two-sided):
    Odds Ratio : 4.9645
    p-valor    : 0.133010

  Conclusão: Diferença NÃO significativa estatisticamente
  (p = 0.133010 ≥ α = 0.05)


In [18]:
# ─── 9. FISHER POR TRIMESTRE — por instituição (biênio agregado) ─────────────
# Cada trimestre (Q1/Q2/Q3/Q4) agrega os dois anos do biênio.
# O Fisher compara Masculino vs Feminino DENTRO de cada trimestre,
# separadamente para FAMAR e FUMES.

def fisher_por_trimestre_bienio(df: pd.DataFrame, nome_inst: str):
    df = df.copy()
    df['genero_norm'] = normalizar_genero(df['genero'])

    # Extrai apenas o número do trimestre (Q1, Q2, Q3, Q4) ignorando o ano
    df['trim_bienio'] = df['data_do_acidente'].dt.quarter.map(
        {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
    )

    contagem = (
        df[df['genero_norm'].notna()]
          .groupby(['trim_bienio', 'genero_norm'])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=['masculino', 'feminino'], fill_value=0)
    )

    print(f'\n{"-" * 58}')
    print(f'  {nome_inst} — Fisher por Trimestre do Biênio (Masc vs Fem)')
    print(f'{"-" * 58}')
    print(f'  {"Trim":<6} {"Masculino":>12} {"Feminino":>12} {"OR":>8} {"p-valor":>12}')
    print(f'  {"-"*52}')

    for trim in ['Q1', 'Q2', 'Q3', 'Q4']:
        if trim not in contagem.index:
            print(f'  {trim:<6} {"sem dados":>12} {"":>12} {"N/A":>8} {"N/A":>12}')
            continue

        m = int(contagem.loc[trim, 'masculino'])
        f = int(contagem.loc[trim, 'feminino'])
        tot = m + f

        # Fisher exige ao menos 1 caso em cada célula comparável;
        # aqui a tabela é [[M, F], [F_esperado, M_esperado]] —
        # como temos só 1 linha de dados, usamos [[M, F], [F, M]]
        # (tabela de simetria) para testar associação gênero×trimestre.
        # Alternativa correta: comparar cada trimestre contra o restante do biênio.
        resto_m = contagem['masculino'].sum() - m
        resto_f = contagem['feminino'].sum() - f

        tbl = np.array([[m, f], [resto_m, resto_f]])

        if tbl.sum() == 0 or tbl[0].sum() == 0 or tbl[1].sum() == 0:
            pct_m = (m / tot * 100) if tot > 0 else 0
            pct_f = (f / tot * 100) if tot > 0 else 0
            print(f'  {trim:<6} {m:>6} ({pct_m:>4.1f}%) {f:>6} ({pct_f:>4.1f}%) {"N/A":>8} {"N/A":>12}')
            continue

        or_t, p_t = fisher_exact(tbl, alternative='two-sided')
        pct_m = (m / tot * 100) if tot > 0 else 0
        pct_f = (f / tot * 100) if tot > 0 else 0
        sig = ' *' if p_t < 0.05 else ''
        print(f'  {trim:<6} {m:>6} ({pct_m:>4.1f}%) {f:>6} ({pct_f:>4.1f}%) {or_t:>8.3f} {p_t:>11.6f}{sig}')

    total_m = int(contagem['masculino'].sum())
    total_f = int(contagem['feminino'].sum())
    total   = total_m + total_f
    print(f'  {"-"*52}')
    print(f'  {"TOTAL":<6} {total_m:>6} ({total_m/total*100:>4.1f}%) {total_f:>6} ({total_f/total*100:>4.1f}%) biênio completo')

print('=' * 58)
print('  TESTE EXATO DE FISHER POR TRIMESTRE — BIÊNIO AGREGADO')
print('  (cada trimestre vs restante do biênio, por instituição)')
print('=' * 58)

fisher_por_trimestre_bienio(df_famar, 'FAMAR')
fisher_por_trimestre_bienio(df_fumes, 'FUMES')

print('\n  * p < 0.05 (significativo)   OR = Odds Ratio')
print('  Fisher: trimestre selecionado vs demais trimestres do biênio')

  TESTE EXATO DE FISHER POR TRIMESTRE — BIÊNIO AGREGADO
  (cada trimestre vs restante do biênio, por instituição)

----------------------------------------------------------
  FAMAR — Fisher por Trimestre do Biênio (Masc vs Fem)
----------------------------------------------------------
  Trim      Masculino     Feminino       OR      p-valor
  ----------------------------------------------------
  Q1          6 (12.2%)     43 (87.8%)    0.488    0.200654
  Q2          3 ( 8.6%)     32 (91.4%)    0.330    0.093454
  Q3         12 (23.1%)     40 (76.9%)    1.377    0.530755
  Q4         13 (33.3%)     26 (66.7%)    2.738    0.020356 *
  ----------------------------------------------------
  TOTAL      34 (19.4%)    141 (80.6%) biênio completo

----------------------------------------------------------
  FUMES — Fisher por Trimestre do Biênio (Masc vs Fem)
----------------------------------------------------------
  Trim      Masculino     Feminino       OR      p-valor
  ---------------

In [19]:
# ─── 10. RESUMO FINAL ────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  RESUMO EXECUTIVO')
print('=' * 60)

for nome, df in planilhas.items():
    print(f'  {nome:<25}: {len(df):>6} registros')

print(f'  {"-"*45}')
print(f'  {"Biênio FAMAR":<25}: {len(df_famar):>6} registros')
print(f'  {"Biênio FUMES":<25}: {len(df_fumes):>6} registros')
print(f'  {"Total geral":<25}: {len(df_famar)+len(df_fumes):>6} registros')
print(f'\n  Fisher geral (FAMAR vs FUMES):')
print(f'    OR = {odds_ratio:.4f} | p = {p_valor:.6f}')
conclusao = 'SIGNIFICATIVO (p < 0.05)' if p_valor < 0.05 else 'NÃO significativo (p ≥ 0.05)'
print(f'    Resultado: {conclusao}')


  RESUMO EXECUTIVO
  2023_famar.csv           :     93 registros
  2024_famar.csv           :     83 registros
  2023_fumes.csv           :      9 registros
  2024_fumes.csv           :     12 registros
  ---------------------------------------------
  Biênio FAMAR             :    176 registros
  Biênio FUMES             :     21 registros
  Total geral              :    197 registros

  Fisher geral (FAMAR vs FUMES):
    OR = 4.9645 | p = 0.133010
    Resultado: NÃO significativo (p ≥ 0.05)
